# Goal 3 — Tokenizer Grafting: Embedding Surgery

**Objective:** Graft our trained Armenian tokenizer onto a pretrained LM by extending
its vocabulary and reinitializing the embedding table. Compare 3 initialization strategies.

**Base model:** Qwen2.5-0.5B (small, ungated, 152k vocab, 0 Armenian tokens)

**Armenian tokenizer:** Best model from Goal 2 (expected: unigram_16k)

**Hardware:** NVIDIA A100 40GB GPU (Google Cloud)

---

## Overview of the Surgery

```
1. Load base model (Qwen2.5-0.5B) with its tokenizer
2. Load our trained Armenian SentencePiece tokenizer
3. Find Armenian tokens NOT in the base vocabulary
4. Add those new tokens to the base tokenizer
5. Resize model's embedding table + LM head
6. Initialize new embeddings using 3 strategies:
   (a) Mean-init:     new_emb = mean of ALL old embeddings
   (b) Heuristic-init: new_emb = mean of old subtokens that represent this word
   (c) Nearest-init:  new_emb = embedding of most similar existing token
7. Evaluate initial perplexity (before fine-tuning)
8. Save 3 model variants for Goal 4 (recovery fine-tuning)
```

## 0. Setup

In [ ]:
# Install if needed (uncomment on fresh VM)
# !pip install torch transformers sentencepiece protobuf accelerate tqdm tabulate

import torch
import sentencepiece as spm
import os
import json
import time
import copy
import math
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Configuration

```
BASE MODEL CHOICE: Qwen2.5-0.5B
-----------------------------------
Considered:
  (a) Qwen2.5-0.5B  — 0.5B params, 152k vocab, 0 Armenian tokens, ungated
  (b) Gemma-2-2B     — 2B params, 256k vocab, 246 Armenian tokens, gated
  (c) mGPT-Armenian  — 1.3B params, 100k vocab, 0 Armenian tokens, ungated
  (d) Mistral-7B     — 7B params, 32k vocab, 28 Armenian tokens, gated

Chose Qwen2.5-0.5B because:
  - Smallest model (0.5B) = fastest surgery and fine-tuning
  - Zero Armenian tokens = cleanest surgery (no conflicting fragments)
  - Ungated = no auth hassle on Google Cloud VM
  - Good English performance (fertility 1.29) = strong baseline to preserve
  - 152k vocab is large enough to absorb 8-16k new tokens without issues
  - Modern architecture (RoPE, GQA, SwiGLU)

  Gemma-2 was tempting (256k vocab, some Armenian) but gated + 2B params
  means slower experiments. For a 3-day sprint, speed matters.
  
  mGPT-Armenian is strategically interesting (Armenian-tuned weights, broken
  tokenizer) but its GPT-2 architecture is outdated and the 1.3B size is
  slower to iterate on.
```

```
GRAFTING STRATEGY: Vocabulary Extension (not Replacement)
------------------------------------------------------------
Considered:
  (a) EXTEND: Keep old vocab, add new Armenian tokens on top
  (b) REPLACE: Throw away old tokenizer, use only Armenian tokenizer

Chose EXTEND because:
  - Preserves all English/multilingual capability
  - Only new Armenian tokens need embedding initialization
  - Old token embeddings are fully trained and untouched
  - Smaller surgery = less recovery training needed
  - More practical for real-world use (bilingual model)
  - REPLACE would require reinitializing ALL embeddings and
    far more recovery training to regain English performance
```

In [ ]:
# =============================================================
# CONFIGURATION
# =============================================================
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"         # Base LM
ARMENIAN_SP_MODEL = str(REPO_ROOT / "models" / "tokenizers" / "hy_bpe_32k.model")
EVAL_PATH = str(REPO_ROOT / "data" / "sample" / "hy_sample_raw.txt")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "goal3_grafted_models")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Init strategy names
STRATEGIES = ["mean_init", "heuristic_init", "nearest_init"]


## 2. Load Base Model and Tokenizer

In [ ]:
print("Loading base model and tokenizer...")
t0 = time.time()

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float32,    # float32 for surgery precision
    device_map="cpu",             # keep on CPU during surgery
)

elapsed = time.time() - t0
print(f"Loaded in {elapsed:.0f}s")
print(f"Base vocab size: {len(base_tokenizer):,}")
print(f"Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Embedding dim: {base_model.config.hidden_size}")
print(f"Embedding table shape: {base_model.get_input_embeddings().weight.shape}")
print(f"LM head shape: {base_model.lm_head.weight.shape}")

## 3. Load Armenian Tokenizer & Find New Tokens

In [ ]:
# Load our trained SentencePiece model
sp = spm.SentencePieceProcessor()
sp.load(ARMENIAN_SP_MODEL)
print(f"Armenian tokenizer vocab size: {sp.get_piece_size():,}")

# Extract all Armenian token strings
armenian_tokens = []
for i in range(sp.get_piece_size()):
    piece = sp.id_to_piece(i)
    # Skip special tokens and byte fallback tokens
    if piece.startswith("<") and piece.endswith(">"):
        continue
    armenian_tokens.append(piece)

print(f"Armenian tokens (excluding special): {len(armenian_tokens):,}")

# Find tokens that don't exist in the base tokenizer
base_vocab = set(base_tokenizer.get_vocab().keys())
new_tokens = [t for t in armenian_tokens if t not in base_vocab]
existing_tokens = [t for t in armenian_tokens if t in base_vocab]

print(f"\nTokens already in base vocab: {len(existing_tokens):,}")
print(f"NEW tokens to add: {len(new_tokens):,}")
print(f"Post-surgery vocab size: {len(base_tokenizer) + len(new_tokens):,}")

# Show some examples
print(f"\nExample new tokens (first 20): {new_tokens[:20]}")
print(f"Example existing tokens (first 20): {existing_tokens[:20]}")

## 4. Extend the Tokenizer

Add all new Armenian tokens to the base tokenizer.

In [ ]:
# Add new tokens to the base tokenizer
num_added = base_tokenizer.add_tokens(new_tokens)
print(f"Added {num_added:,} new tokens to base tokenizer")
print(f"New tokenizer vocab size: {len(base_tokenizer):,}")

# Verify: encode a sample Armenian text
# (the new tokens should now be used instead of byte fallback)
test_text = new_tokens[0] if new_tokens else "test"
old_enc = base_tokenizer.encode(test_text)
print(f"\nTest encoding of '{test_text}': {old_enc}")

## 5. Embedding Initialization Strategies

```
STRATEGY A — Mean Init (Baseline)
-----------------------------------
new_embedding = mean(all old embeddings)

Simplest approach. Every new token starts at the centroid of the
embedding space. Gives the model a "neutral starting point" — not
biased toward any particular meaning.

Pros: Simple, fast, deterministic
Cons: All new tokens start identical — the model must learn to
      differentiate them entirely during fine-tuning.


STRATEGY B — Heuristic Init (FOCUS-style)
--------------------------------------------
For each new Armenian token t_new:
  1. Encode t_new's surface form using the OLD tokenizer
     e.g., our token "ացում" → old tokenizer splits to [" delays", "ац", "ум"]
  2. Look up old embeddings for those subtokens: [e1, e2, e3]
  3. Average them: new_embedding = mean(e1, e2, e3)

This gives each new token a starting embedding that approximates
its meaning based on how the old model would have processed it.

Pros: Semantically informed initialization — each token starts
      close to where it should end up. Less fine-tuning needed.
Cons: Relies on the old tokenizer's (poor) Armenian fragmentation
      to provide meaningful subtokens. If the old tokenizer splits
      Armenian into random bytes, the averages may be noisy.

This is essentially the FOCUS method (Dobler & de Melo, 2023).


STRATEGY C — Nearest Token Init
----------------------------------
For each new Armenian token t_new:
  1. Compute a "target" embedding using the heuristic method above
  2. Find the existing token whose embedding is nearest (cosine sim)
  3. Copy that existing token's embedding

This avoids the averaging problem — instead of a blurred average,
you get a sharp, well-trained embedding from a related token.

Pros: Each new embedding is a real, well-trained embedding point
Cons: May not find truly relevant neighbors if the old model has
      no Armenian knowledge. Also slower to compute.
```

In [ ]:
def get_old_embeddings(model):
    """Extract input embeddings and LM head weights from the model."""
    input_embeds = model.get_input_embeddings().weight.data.clone()
    lm_head_weights = model.lm_head.weight.data.clone()
    return input_embeds, lm_head_weights


def mean_init(old_embeds, num_new_tokens):
    """
    Strategy A: Initialize all new tokens as the mean of all old embeddings.
    """
    mean_emb = old_embeds.mean(dim=0)  # shape: (hidden_dim,)
    new_embeds = mean_emb.unsqueeze(0).expand(num_new_tokens, -1).clone()
    # Add small random noise to break symmetry
    noise = torch.randn_like(new_embeds) * 0.01
    return new_embeds + noise


def heuristic_init(old_embeds, new_tokens_list, base_tok, old_vocab_size):
    """
    Strategy B: Initialize each new token as the average of old subtokens
    that the base tokenizer would use to represent it (FOCUS method).
    """
    hidden_dim = old_embeds.shape[1]
    new_embeds = torch.zeros(len(new_tokens_list), hidden_dim)
    
    fallback_mean = old_embeds[:old_vocab_size].mean(dim=0)
    
    for i, token_str in enumerate(new_tokens_list):
        # Decode the SentencePiece representation to get the surface form
        # Remove the leading space marker if present
        surface = token_str.replace("\u2581", " ").strip()
        if not surface:
            new_embeds[i] = fallback_mean
            continue
        
        # Encode using the OLD tokenizer (before we added new tokens)
        # This gives us the byte-fallback encoding the old model would use
        old_ids = base_tok.encode(surface, add_special_tokens=False)
        
        if not old_ids:
            new_embeds[i] = fallback_mean
            continue
        
        # Average the old embeddings for these subtokens
        # Only use IDs within the old vocab range
        valid_ids = [t for t in old_ids if t < old_vocab_size]
        if valid_ids:
            subtoken_embeds = old_embeds[valid_ids]  # shape: (n_subtokens, hidden_dim)
            new_embeds[i] = subtoken_embeds.mean(dim=0)
        else:
            new_embeds[i] = fallback_mean
    
    return new_embeds


def nearest_init(old_embeds, new_tokens_list, base_tok, old_vocab_size):
    """
    Strategy C: For each new token, find the nearest existing token
    (by cosine similarity to its heuristic embedding) and copy it.
    """
    # First compute heuristic embeddings as targets
    heuristic_embeds = heuristic_init(
        old_embeds, new_tokens_list, base_tok, old_vocab_size
    )
    
    # Normalize old embeddings for cosine similarity
    old_subset = old_embeds[:old_vocab_size]  # only original vocab
    old_norms = torch.nn.functional.normalize(old_subset, dim=1)  # (old_vocab, hidden)
    
    # Find nearest neighbor for each new token
    new_embeds = torch.zeros_like(heuristic_embeds)
    
    # Process in batches to avoid OOM
    batch_size = 512
    for start in range(0, len(new_tokens_list), batch_size):
        end = min(start + batch_size, len(new_tokens_list))
        batch = heuristic_embeds[start:end]
        batch_norms = torch.nn.functional.normalize(batch, dim=1)
        
        # Cosine similarity: (batch_size, old_vocab_size)
        sims = torch.mm(batch_norms, old_norms.t())
        nearest_ids = sims.argmax(dim=1)
        
        new_embeds[start:end] = old_subset[nearest_ids]
    
    return new_embeds

## 6. Perform the Surgery

Create 3 variants of the model — one for each init strategy.

In [ ]:
def graft_model(base_model, base_tokenizer, new_tokens_list, strategy_name, strategy_fn, output_dir):
    """
    Perform tokenizer surgery on a model:
    1. Resize embedding table + LM head
    2. Initialize new embeddings using the given strategy
    3. Save the model
    """
    print(f"\n{'='*60}")
    print(f"Grafting with strategy: {strategy_name}")
    print(f"{'='*60}")
    
    # Deep copy the model so each strategy gets a fresh copy
    print("  Cloning model...")
    model = copy.deepcopy(base_model)
    
    # Get old embeddings BEFORE resizing
    old_vocab_size = model.get_input_embeddings().weight.shape[0]
    old_input_embeds, old_lm_head = get_old_embeddings(model)
    print(f"  Old vocab size: {old_vocab_size:,}")
    
    # Resize the model's embedding table and LM head
    new_vocab_size = old_vocab_size + len(new_tokens_list)
    model.resize_token_embeddings(new_vocab_size)
    print(f"  New vocab size: {new_vocab_size:,} (+{len(new_tokens_list):,})")
    
    # Compute new embeddings using the strategy
    print(f"  Computing {strategy_name} embeddings...")
    t0 = time.time()
    
    if strategy_name == "mean_init":
        new_embeds = strategy_fn(old_input_embeds, len(new_tokens_list))
    else:
        new_embeds = strategy_fn(
            old_input_embeds, new_tokens_list, base_tokenizer, old_vocab_size
        )
    
    elapsed = time.time() - t0
    print(f"  Computed in {elapsed:.1f}s")
    
    # Write new embeddings into the model
    with torch.no_grad():
        # Input embeddings
        model.get_input_embeddings().weight[old_vocab_size:] = new_embeds
        
        # LM head (output projection)
        # For models with tied embeddings, this may already be updated.
        # For models with untied heads, we initialize the LM head too.
        if not model.config.tie_word_embeddings:
            model.lm_head.weight.data[old_vocab_size:] = new_embeds.clone()
            print("  Initialized untied LM head separately")
        else:
            print("  Embeddings are tied — LM head auto-updated")
    
    # Verify shapes
    print(f"  Input embeddings shape: {model.get_input_embeddings().weight.shape}")
    print(f"  LM head shape: {model.lm_head.weight.shape}")
    
    # Save
    save_dir = os.path.join(output_dir, strategy_name)
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    base_tokenizer.save_pretrained(save_dir)
    print(f"  Saved to: {save_dir}/")
    
    return model, save_dir

In [ ]:
# We need a copy of the tokenizer BEFORE adding new tokens
# for the heuristic and nearest strategies (they need to encode
# Armenian text using the OLD tokenizer to get subtoken IDs)
old_tokenizer_for_init = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

# Strategy function mapping
strategy_fns = {
    "mean_init": mean_init,
    "heuristic_init": heuristic_init,
    "nearest_init": nearest_init,
}

# Run all 3 strategies
grafted_models = {}
grafted_dirs = {}

for strategy_name in STRATEGIES:
    model, save_dir = graft_model(
        base_model=base_model,
        base_tokenizer=old_tokenizer_for_init,
        new_tokens_list=new_tokens,
        strategy_name=strategy_name,
        strategy_fn=strategy_fns[strategy_name],
        output_dir=OUTPUT_DIR,
    )
    grafted_models[strategy_name] = model
    grafted_dirs[strategy_name] = save_dir

print(f"\nAll 3 grafted models saved!")
for name, d in grafted_dirs.items():
    print(f"  {name}: {d}")

## 7. Evaluate Initial Perplexity (Before Fine-tuning)

This measures how well each initialization strategy performs
*before* any recovery training. Lower perplexity = better.

We expect:
- All three to have very high perplexity (the model hasn't learned Armenian yet)
- Heuristic init to be slightly better than mean init
- Nearest init to be similar to heuristic
- The gap between strategies will narrow after fine-tuning (Goal 4)

In [ ]:
import unicodedata

def is_armenian_char(c):
    cp = ord(c)
    return (0x0530 <= cp <= 0x058F) or (0xFB13 <= cp <= 0xFB17)

def load_eval_texts(path, max_lines=2000):
    """Load evaluation lines."""
    lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            line = unicodedata.normalize("NFC", line.strip())
            if len(line) < 20:
                continue
            arm_count = sum(1 for c in line if is_armenian_char(c))
            if arm_count >= 10:
                lines.append(line)
    return lines

eval_texts = load_eval_texts(EVAL_PATH)
print(f"Loaded {len(eval_texts)} evaluation lines")

In [ ]:
def compute_perplexity(model, tokenizer, texts, max_length=512, batch_size=8):
    """
    Compute perplexity of a model on given texts.
    Uses sliding window approach for texts longer than max_length.
    """
    model.eval()
    model.to(DEVICE)
    
    total_loss = 0.0
    total_tokens = 0
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            encodings = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=True,
            )
            
            input_ids = encodings.input_ids.to(DEVICE)
            attention_mask = encodings.attention_mask.to(DEVICE)
            
            # Labels = input_ids (auto-regressive LM)
            labels = input_ids.clone()
            # Mask padding tokens in labels
            labels[attention_mask == 0] = -100
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            
            # Loss is already averaged over non-masked tokens
            num_tokens = (labels != -100).sum().item()
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    
    model.cpu()  # free GPU for next model
    torch.cuda.empty_cache()
    
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float("inf")
    perplexity = math.exp(avg_loss) if avg_loss < 100 else float("inf")
    
    return perplexity, avg_loss, total_tokens

In [ ]:
# Load the extended tokenizer (with Armenian tokens added)
# All 3 models use the same tokenizer
extended_tokenizer = AutoTokenizer.from_pretrained(
    grafted_dirs["mean_init"], trust_remote_code=True
)

# Evaluate each strategy
print("Evaluating perplexity (before fine-tuning)...")
print("This measures the STARTING POINT for each strategy.\n")

ppl_results = {}
for strategy_name, model in grafted_models.items():
    print(f"  {strategy_name}...")
    t0 = time.time()
    ppl, loss, n_tokens = compute_perplexity(model, extended_tokenizer, eval_texts)
    elapsed = time.time() - t0
    ppl_results[strategy_name] = {"perplexity": ppl, "loss": loss, "tokens": n_tokens}
    print(f"    PPL={ppl:.2f}, loss={loss:.4f}, tokens={n_tokens:,} ({elapsed:.0f}s)")

# Also evaluate the original model (no surgery) as reference
print(f"  original (no surgery)...")
original_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
ppl, loss, n_tokens = compute_perplexity(base_model, original_tokenizer, eval_texts)
ppl_results["original"] = {"perplexity": ppl, "loss": loss, "tokens": n_tokens}
print(f"    PPL={ppl:.2f}, loss={loss:.4f}, tokens={n_tokens:,}")

In [ ]:
# Summary table
print("\n" + "=" * 60)
print("PERPLEXITY COMPARISON (before fine-tuning)")
print("=" * 60)
print(f"{'Strategy':<20} {'Perplexity':>12} {'Loss':>10} {'Tokens':>10}")
print("-" * 55)
for name in ["original"] + STRATEGIES:
    r = ppl_results[name]
    ppl_str = f"{r['perplexity']:.2f}" if r['perplexity'] < 1e6 else "inf"
    print(f"{name:<20} {ppl_str:>12} {r['loss']:>10.4f} {r['tokens']:>10,}")

print("\nNote: The original model uses byte-fallback for Armenian,")
print("so it processes many more tokens for the same text.")
print("After fine-tuning (Goal 4), grafted models should achieve")
print("much lower perplexity than the original on Armenian text.")

## 8. Verify Tokenization Improvement

In [ ]:
# Compare tokenization: before vs after surgery
sample_text = eval_texts[0] if eval_texts else "Test"

print(f"Sample text: {sample_text}\n")

# Original tokenizer
orig_tokens = original_tokenizer.encode(sample_text, add_special_tokens=False)
orig_decoded = [original_tokenizer.decode([t]) for t in orig_tokens]
print(f"BEFORE surgery ({len(orig_tokens)} tokens):")
print(f"  {orig_decoded[:30]}{'...' if len(orig_tokens) > 30 else ''}")

# Extended tokenizer
ext_tokens = extended_tokenizer.encode(sample_text, add_special_tokens=False)
ext_decoded = [extended_tokenizer.decode([t]) for t in ext_tokens]
print(f"\nAFTER surgery ({len(ext_tokens)} tokens):")
print(f"  {ext_decoded[:30]}{'...' if len(ext_tokens) > 30 else ''}")

reduction = (1 - len(ext_tokens) / len(orig_tokens)) * 100
print(f"\nToken count reduction: {reduction:.1f}%")
print(f"Fertility improvement: {len(orig_tokens)/len(ext_tokens):.1f}x fewer tokens")

## 9. Compute Fertility Metrics on Extended Tokenizer

In [ ]:
def compute_fertility(tokenizer, texts):
    total_tokens = 0
    total_words = 0
    total_bytes = 0
    single_token_words = 0
    
    for text in texts:
        words = [w for w in text.split() if any(c.isalpha() for c in w)]
        if not words:
            continue
        tokens = tokenizer.encode(text, add_special_tokens=False)
        total_tokens += len(tokens)
        total_bytes += len(text.encode("utf-8"))
        total_words += len(words)
        for w in words:
            wt = tokenizer.encode(w, add_special_tokens=False)
            if len(wt) == 1:
                single_token_words += 1
    
    return {
        "fertility": round(total_tokens / total_words, 3),
        "bytes_per_token": round(total_bytes / total_tokens, 3),
        "strr": round(single_token_words / total_words, 4),
    }

# Compare original vs extended tokenizer fertility
orig_fert = compute_fertility(original_tokenizer, eval_texts)
ext_fert = compute_fertility(extended_tokenizer, eval_texts)

print("Fertility Comparison:")
print(f"  Original Qwen2.5: fertility={orig_fert['fertility']}, "
      f"bytes/tok={orig_fert['bytes_per_token']}, STRR={orig_fert['strr']}")
print(f"  Extended (grafted): fertility={ext_fert['fertility']}, "
      f"bytes/tok={ext_fert['bytes_per_token']}, STRR={ext_fert['strr']}")
print(f"  Improvement: {orig_fert['fertility']/ext_fert['fertility']:.1f}x better fertility")

## 10. Save Results & Prepare for Goal 4

In [ ]:
# Save Goal 3 results
goal3_results = {
    "base_model": BASE_MODEL_NAME,
    "armenian_tokenizer": ARMENIAN_SP_MODEL,
    "new_tokens_added": len(new_tokens),
    "original_vocab_size": base_model.get_input_embeddings().weight.shape[0],
    "extended_vocab_size": len(extended_tokenizer),
    "perplexity_before_finetuning": ppl_results,
    "fertility_original": orig_fert,
    "fertility_extended": ext_fert,
    "grafted_model_dirs": grafted_dirs,
    "strategies": STRATEGIES,
}

results_path = os.path.join(OUTPUT_DIR, "goal3_results.json")
with open(results_path, "w") as f:
    json.dump(goal3_results, f, indent=2, default=str)

print(f"Goal 3 results saved to: {results_path}")
print(f"\nGrafted model directories (for Goal 4 fine-tuning):")
for name, d in grafted_dirs.items():
    print(f"  {name}: {d}/")
    
print(f"\n--- Goal 3 complete! Each directory contains a full model + tokenizer.")
print(f"--- Proceed to Goal 4: Recovery fine-tuning with LoRA on Armenian corpus.")